## "openai/gpt-oss-120b"— Cloud Pipeline
Aspect-Based Sentiment Analysis using the Groq API.
Processes restaurant reviews and extracts structured sentiment scores
across 5 aspects: overall, food, service, price, ambiance.

**Requirements:** Groq API key, `"openai/gpt-oss-120b"` and .env file

In [1]:
import pandas as pd
import json
import time
import os
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel, Field

In [ ]:
# 1. Load the variables from the .env file
# This looks for a file named .env in the current folder and loads the variables
load_dotenv()

# 2. Fetch the key safely
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# 3. Check if it worked
if not GROQ_API_KEY:
    print("❌ Error: API Key not found.")
    print("Did you create the .env file and save it in the same folder?")
else:
    print("✅ Key found! Configuring client...")
    client = Groq(api_key=GROQ_API_KEY)

In [12]:
# Define the structure for a single aspect
class AspectScore(BaseModel):
    score: float | None = None
    quotes: list[str] | None = None

# Define the overall structure we expect the model to return
class ReviewAnalysis(BaseModel):
    overall: AspectScore = AspectScore()
    food: AspectScore = AspectScore()
    service: AspectScore = AspectScore()
    price: AspectScore = AspectScore()
    ambiance: AspectScore = AspectScore()

In [13]:
# Define the expected structure for Groq's metadata, mirroring Ollama's setup
class GroqUsage(BaseModel):
    input_tokens: int | None = Field(default=None, alias="prompt_tokens")
    output_tokens: int | None = Field(default=None, alias="completion_tokens")
    total_tokens: int | None = Field(default=None)
    total_time: float | None = Field(default=None)

class GroqMetadata(BaseModel):
    model_name: str | None = Field(default=None, alias="model")
    created_at_unix: int | None = Field(default=None, alias="created")
    usage: GroqUsage = Field(default_factory=GroqUsage)

In [14]:
schema_instructions = ReviewAnalysis.model_json_schema()

SYSTEM_PROMPT = f"""
[Context]

You are an expert restaurant critic performing structured data analysis to identify performance issues.

[Task]

Your job is to perform Aspect-Based Sentiment Analysis on the restaurant reviews. You must evaluate the customers sentiment regarding five specific aspects: "overall", "food", "service", "price" and "ambiance".

[Constraints]

For each category (overall, food, service, price, ambiance), assign a score:
-1.0 (Strong Negative)
-0.5 (Negative)
 0.0 (Neutral / Mixed)
+0.5 (Positive)
+1.0 (Strong Positive)
null (Not Mentioned)

You can use every number in between. But NEVER use a number lower than -1.0 or higher than 1.0.

For each aspect, provide quotes: a short verbatim excerpt from the review that justifies the score. If an aspect is not mentioned, set both score and quote to null.

The "quotes" field must be either:
- a list of strings (even if only one quote is provided), or
- null

Only assign a score if the review explicitly mentions that aspect. Do NOT infer or assume sentiment that is not directly stated. The "overall" aspect is an exception — it reflects your holistic reading of the review and may not have a direct quote, in which case set quote to null.
The overall category is not allowed to be null or empty, always add your evalution there.

Pay close attention to sarcasm. If the tone contradicts the actual event being described, adjust the score accordingly.

[Format]

Output ONLY valid JSON that strictly matches this schema:
{json.dumps(schema_instructions, indent=2)}

[Few-Shot Examples]

Example 1:

Review: "The fried chicken taste great and the fries were very fresh and the customer service was good. I can't complain. I have not had fried chicken in Chicago that taste like that!"

Output: {{
  "overall": {{"score": 0.7, "quotes": ["I can't complain"]}},
  "food": {{"score": 1.0, "quotes": ["the fried chicken taste great", "I have not had fried chicken in Chicago that taste like that"]}},
  "service": {{"score": 0.5, "quotes": ["customer service was good"]}},
  "price": {{"score": null, "quotes": null}},
  "ambiance": {{"score": null, "quotes": null}}
}}

Example 2:

Review: "One of the worst meals I have ever eaten, I was told this was an iconic place to eat do not go there. The place was dirty the food was cold orders were wrong. Had to get our own silverware and refills on drinks. They said it's a thirty minute wait on fried chicken dinner so you expect hot juicy chicken wrong!! Cold dry chicken greens had no flavor cooked in water and no seasoning. the best thing-in this restaurant is the exit door save your money."

Output: {{
  "overall": {{"score": -1.0, "quotes": ["the best thing-in this restaurant is the exit door save your money"]}},
  "food": {{"score": -1.0, "quotes": ["One of the worst meals I have ever eaten", "Cold dry chicken greens had no flavor cooked in water and no seasoning"]}},
  "service": {{"score": -0.2, "quotes": ["orders were wrong"]}},
  "price": {{"score": null, "quotes": null}},
  "ambiance": {{"score": -0.7, "quotes": ["the place was dirty"]}}
}}

Example 3:

Review: "A group of 6 of us were visiting had we heard good things about Mother's.  We had the Ferdi, fried chicken, gumbo and a couple sides. Everyone enjoyed their meals. Good service, good value for the amount of food. Would go back again."

Output: {{
  "overall":  {{"score": 0.7, "quotes": ["Would go back again."]}},
  "food":     {{"score": 0.7, "quotes": ["Everyone enjoyed their meals"]}},
  "service":  {{"score": 0.7, "quotes": ["Good service"]}},
  "price":    {{"score": 0.7, "quotes": ["good value for the amount of food"]}},
  "ambiance": {{"score": null, "quotes": null}}
}}

Example 4:

Review: "The service here is unbelievable. Everyone just treated you like family here. braised greens and the fried chicken are both a must try here. At the end of the day, you are not allowed to tip anyone here."

Output: {{
  "overall": {{"score": 0.8, "quotes": null}},
  "food": {{"score": 0.7, "quotes": ["braised greens and the fried chicken are both a must try here"]}},
  "service": {{"score": 1.0, "quotes": ["The service here is unbelievable", "Everyone just treated you like family here"]}},
  "price": {{"score": null, "quotes": null}},
  "ambiance": {{"score": null, "quotes": null}}
}}

Example 5: Review: "Oh fantastic, waited an hour for lukewarm food. Really worth the trip."

Output: {{
  "overall": {{"score": -0.8, "quotes": null}},
  "food": {{"score": -1.0, "quotes": ["waited an hour for lukewarm food"]}},
  "service": {{"score": -0.5, "quotes": ["waited an hour"]}},
  "price": {{"score": null, "quotes": null}},
  "ambiance": {{"score": null, "quotes": null}}
}}

"""

In [ ]:
def analyse_review_groq(review_text, review_id, retries=3):
    """
    Calls Groq API with a retry loop.
    Matches the logic of analyse_review_local.
    """
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": review_text}
                ],
                model="openai/gpt-oss-120b",
                temperature=0.0,
                response_format={"type": "json_object"}
            )
            return response

        except Exception as e:
            print(f"Attempt {attempt + 1} failed for review {review_id}: {e}")
            if attempt < retries - 1:
                time.sleep(10)
            else:
                print(f"All {retries} Groq attempts failed for review {review_id}.")
                return None

In [27]:
def parse_groq_metadata(response, review_id):
    """
    Extracts the metadata using Pydantic.
    Matches the robust parsing style of the local pipeline.
    """
    # 1. Instantly create an empty fallback structure
    fallback_data = GroqMetadata().model_dump()
    # Flatten the usage dictionary for pandas
    fallback_data.update(fallback_data.pop('usage'))
    fallback_data["review_id"] = review_id

    if response is None:
        return fallback_data

    try:
        # 2. Validate the object returned by Groq using Pydantic
        # Convert response object to dict first, as Groq returns a custom object
        resp_dict = response.model_dump()
        validated_meta = GroqMetadata.model_validate(resp_dict)

        # 3. Convert to dict, flatten the nested usage stats, and append ID
        final_dict = validated_meta.model_dump()
        final_dict.update(final_dict.pop('usage'))
        final_dict["review_id"] = review_id

        return final_dict

    except Exception as e:
        print(f"Error parsing Groq metadata for {review_id}: {e}")
        return fallback_data

In [28]:
def parse_groq_sentiment(response, review_id):
    """
    Validates JSON content from Groq response using Pydantic.
    """
    # Create a completely empty default response just in case the API completely fails
    fallback_data = ReviewAnalysis().model_dump()
    fallback_data["review_id"] = review_id

    if response is None:
        return fallback_data

    try:
        content_str = response.choices[0].message.content

        # Pydantic validates the string, enforces types, and maps it to the object
        validated_data = ReviewAnalysis.model_validate_json(content_str)

        # Convert it to a dictionary so Pandas can easily read it later
        final_dict = validated_data.model_dump()
        final_dict["review_id"] = review_id

        return final_dict

    except Exception as e:
        print(f"Error parsing Groq sentiment for review {review_id}: {e}")
        return fallback_data

In [29]:
def process_reviews_groq(df):
    """
    Calls the API. Parses the metadata and aspects.
    Returns 2 DataFrames: metadata and aspects.
    """
    sentiment_results = []
    metadata_results = []

    for index, row in df.iterrows():
        r_id = row['review_id']
        text = row['text']

        # 1. Get raw response with retry logic
        api_response = analyse_review_groq(text, r_id)

        # 2. Extract Data
        sentiment = parse_groq_sentiment(api_response, r_id)
        meta = parse_groq_metadata(api_response, r_id)

        sentiment_results.append(sentiment)
        metadata_results.append(meta)

        # Rate Limit Protection
        time.sleep(10)

    # --- Create DataFrames ---
    df_sentiment = pd.json_normalize(sentiment_results, sep='_')
    if not df_sentiment.empty:
        df_sentiment = df_sentiment.set_index('review_id')

    df_meta = pd.DataFrame(metadata_results)
    if not df_meta.empty:
        df_meta = df_meta.set_index('review_id')

    return df_sentiment, df_meta

In [ ]:
file_path = "../data/restaurant_reviews_sample.csv"

# Load the CSV into a DataFrame
data = pd.read_csv(file_path)

# Display the first 5 rows to verify
print(data.head())

In [ ]:
# Run the processing function
df_sentiment_groq, df_meta_groq = process_reviews_groq(data)

In [ ]:
df_meta_groq

In [ ]:
df_sentiment_groq

In [ ]:
df_sentiment_groq.to_csv('../data/groq_aspects_gpt.csv', index=False)
df_meta_groq.to_csv('../data/groq_metadata_gpt.csv', index=False)